# Experiment 5.2.2 — Frozen Local Multi-`tau_syn` Decoder

Analysis-only notebook. Temporal-profile selection uses mean validation native balanced accuracy only; test metrics are reported after selection. `theta=0.5` is fixed and is never used as a sweep variable.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

repo_root = Path.cwd().resolve()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent
root = repo_root / 'notebooks' / 'artifacts' / 'experiment_5_2_2_frozen_local_multitau_syn' / 'frozen_exp51_l2_multitau_syn_wholecount_v1'
runs = pd.read_csv(root / 'runs.csv')
ablations = pd.read_csv(root / 'ablation_runs.csv')
probes = pd.read_csv(root / 'probes.csv')
activity = pd.read_csv(root / 'activity.csv')
threshold_activity = pd.read_csv(root / 'threshold_activity.csv')
histories = pd.read_csv(root / 'histories.csv')
local_reference = pd.read_csv(root / 'local_reference.csv')
manifest = json.loads((root / 'manifest.json').read_text(encoding='utf-8'))
manifest

## Validation-only profile selection

In [ ]:
profile_order = ['s2','s3','s4','s5','s6','s7','s234567','s4567','s567','s67']
native_summary = runs.groupby(['readout','profile']).agg(val_ba_mean=('native_val_balanced_accuracy','mean'), val_ba_sd=('native_val_balanced_accuracy','std'), test_ba_mean=('native_test_balanced_accuracy','mean'), test_ba_sd=('native_test_balanced_accuracy','std')).reset_index()
selection = native_summary.sort_values(['readout','val_ba_mean','profile'], ascending=[True,False,True]).groupby('readout', as_index=False).first()
display(native_summary.sort_values(['readout','profile']))
display(selection)

## Native profile sweep

In [ ]:
fig, ax = plt.subplots(figsize=(11,5))
x = np.arange(len(profile_order))
for readout, group in native_summary.groupby('readout'):
    ordered = group.set_index('profile').reindex(profile_order)
    ax.errorbar(x, ordered['test_ba_mean'], yerr=ordered['test_ba_sd'], marker='o', label=readout)
ax.set_xticks(x, profile_order, rotation=35)
ax.set_ylabel('Test balanced accuracy')
ax.set_xlabel('L3 tau_syn profile')
ax.set_title('Exp5.2.2 native performance across L3 temporal bases')
ax.legend()
fig.tight_layout()

## Frozen L2 to selected L3 representation accessibility

In [ ]:
local_map = {'local_whole_count':'WholeCount','local_fixed250_ordered':'Fixed250','local_relative10_ordered':'Relative10'}
l3_map = {'l3_whole_count':'WholeCount','l3_fixed250_ordered':'Fixed250','l3_relative10_ordered':'Relative10'}
local_mean = local_reference.assign(stage='Frozen L2', metric=local_reference['probe_type'].map(local_map)).groupby(['stage','metric'], as_index=False)['test_ba'].mean()
selected_probe_rows = []
for _, picked in selection.iterrows():
    subset = probes[(probes.readout == picked.readout) & (probes.profile == picked.profile) & (probes.intervention == 'normal')].copy()
    subset['stage'] = f'L3 {picked.readout} / {picked.profile}'
    subset['metric'] = subset['probe_type'].map(l3_map)
    selected_probe_rows.append(subset[['stage','metric','test_ba']])
representation_flow = pd.concat([local_mean, *selected_probe_rows], ignore_index=True)
display(representation_flow.groupby(['stage','metric'], as_index=False).test_ba.mean())

## Reset effects and organization_gap

In [ ]:
reset_order = ['resetall250','reset234','reset2345','reset23456','reset67','reset567']
normal_native = ablations[ablations.intervention == 'normal'][['profile','readout','seed','native_test_balanced_accuracy']].rename(columns={'native_test_balanced_accuracy':'native_normal'})
reset_native = ablations[ablations.intervention != 'normal'].merge(normal_native, on=['profile','readout','seed'], how='left')
reset_native['delta_native_test_ba'] = reset_native['native_normal'] - reset_native['native_test_balanced_accuracy']
probe_wide = probes.pivot_table(index=['profile','readout','seed','intervention'], columns='probe_type', values='test_ba').reset_index()
normal_probe = probe_wide[probe_wide.intervention == 'normal'].drop(columns='intervention').rename(columns={'l3_whole_count':'wc_normal','l3_fixed250_ordered':'fixed_normal','l3_relative10_ordered':'rel_normal'})
reset_probe = probe_wide[probe_wide.intervention != 'normal'].merge(normal_probe, on=['profile','readout','seed'], how='left')
reset_probe['delta_whole_count'] = reset_probe['wc_normal'] - reset_probe['l3_whole_count']
reset_probe['delta_fixed250'] = reset_probe['fixed_normal'] - reset_probe['l3_fixed250_ordered']
reset_probe['delta_relative10'] = reset_probe['rel_normal'] - reset_probe['l3_relative10_ordered']
reset_probe['organization_gap'] = reset_probe['delta_whole_count'] - reset_probe['delta_fixed250']
display(reset_native.groupby(['readout','profile','intervention']).delta_native_test_ba.agg(['mean','std']))
display(reset_probe.groupby(['readout','profile','intervention'])[['delta_whole_count','delta_fixed250','delta_relative10','organization_gap']].mean())

## Output-LIF bottleneck

In [ ]:
output_pick = selection[selection.readout == 'output_lif'].iloc[0]
output_profile = output_pick.profile
output_ab = ablations[(ablations.readout == 'output_lif') & (ablations.profile == output_profile)]
output_probe = probes[(probes.readout == 'output_lif') & (probes.profile == output_profile) & (probes.probe_type == 'l3_whole_count')][['seed','intervention','test_ba']].rename(columns={'test_ba':'fresh_count_probe_test_ba'})
bottleneck = output_ab.merge(output_probe, on=['seed','intervention'], how='left')
display(bottleneck.groupby('intervention')[['fresh_count_probe_test_ba','pre_lif_test_balanced_accuracy','native_test_balanced_accuracy']].mean())

## Per-timescale activity and fixed-threshold operating point

In [ ]:
selected_profiles = set(selection.profile)
activity_selected = activity[activity.profile.isin(selected_profiles) & activity.intervention.isin(['normal','resetall250','reset67','reset567']) & (activity['split'] == 'test') & (activity['shift'].astype(str) != 'all')].copy()
activity_summary = activity_selected.groupby(['readout','profile','intervention','shift']).agg(event_rate_hz=('event_rate_hz','mean'), nonzero_fraction=('nonzero_fraction','mean'), mean_abs_syn=('mean_abs_syn','mean'), rms_syn=('rms_syn','mean'), mean_abs_mem=('mean_abs_mem','mean')).reset_index()
display(activity_summary)
threshold_selected = threshold_activity[threshold_activity.profile.isin(selected_profiles) & threshold_activity.intervention.isin(['normal','resetall250','reset67','reset567']) & (threshold_activity['split'] == 'test') & (threshold_activity['shift'].astype(str) != 'all')].copy()
threshold_summary = threshold_selected.groupby(['readout','profile','intervention','shift']).agg(firing_rate_hz=('firing_rate_hz','mean'), spike_probability=('spike_probability','mean'), mean_abs_input_current=('mean_abs_input_current','mean'), mean_abs_pre_reset_membrane=('mean_abs_pre_reset_membrane','mean'), pre_reset_above_threshold_probability=('pre_reset_above_threshold_probability','mean')).reset_index()
display(threshold_summary)
long_operating_point = threshold_summary[threshold_summary['shift'].astype(str).isin(['6','7'])].copy()
display(long_operating_point)
for (readout, profile), group in threshold_summary.groupby(['readout','profile']):
    fig, ax = plt.subplots(figsize=(8,4))
    for intervention, part in group.groupby('intervention'):
        part = part.sort_values('shift', key=lambda s: s.astype(int))
        ax.plot(part['shift'].astype(str), part['firing_rate_hz'], marker='o', label=intervention)
    ax.set_title(f'L3 per-shift firing at fixed theta=0.5: {readout} / {profile}')
    ax.set_xlabel('shift_syn')
    ax.set_ylabel('FR_s (Hz)')
    ax.legend()
    fig.tight_layout()

The fixed-threshold table reports `FR_s`, `P(S_t=1)`, `E|I_t|`, `E|U_t^-|`, and `P(U_t^- > theta)` per `tau_syn` group. Shifts 6 and 7 are the primary long-timescale operating-point diagnostics.

## Paired effects and selected learning curves

In [ ]:
paired_readout = runs.pivot_table(index=['profile','seed'], columns='readout', values='native_test_balanced_accuracy').dropna().reset_index()
paired_readout['hidden_count_minus_output_lif'] = paired_readout['hidden_count_linear'] - paired_readout['output_lif']
display(paired_readout.groupby('profile').hidden_count_minus_output_lif.agg(['mean','std','min','max']))
for _, picked in selection.iterrows():
    curve = histories[(histories.readout == picked.readout) & (histories.profile == picked.profile)]
    summary = curve.groupby('epoch').val_balanced_accuracy.agg(['mean','std']).reset_index()
    fig, ax = plt.subplots(figsize=(9,4))
    ax.plot(summary['epoch'], summary['mean'])
    ax.fill_between(summary['epoch'], summary['mean'] - summary['std'].fillna(0), summary['mean'] + summary['std'].fillna(0), alpha=0.2)
    ax.set_title(f'Validation-selected: {picked.readout} / {picked.profile}')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Validation balanced accuracy')
    fig.tight_layout()